In [1]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

test_df = pd.read_parquet("../data/processed/test_preds.parquet")

analyzer = SentimentIntensityAnalyzer()

# compound score for every test review
test_df["vader_score"] = test_df["text"].map(
    lambda t: analyzer.polarity_scores(t)["compound"]
)

# binary decision: >= 0 -> positive
test_df["pred_vader"] = (test_df["vader_score"] >= 0).map(
    {True: "positive", False: "negative"}
)

y_test = test_df["sentiment"]
pred = test_df["pred_vader"]

print("Accuracy:", accuracy_score(y_test, pred))
print("F1 (macro):", f1_score(y_test, pred, average="macro"))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred, labels=["negative", "positive"]))

Accuracy: 0.7189
F1 (macro): 0.7041201350142663
              precision    recall  f1-score   support

    negative       0.87      0.50      0.64      4934
    positive       0.66      0.93      0.77      5066

    accuracy                           0.72     10000
   macro avg       0.77      0.72      0.70     10000
weighted avg       0.76      0.72      0.70     10000

[[2477 2457]
 [ 354 4712]]


In [2]:
test_df.to_parquet("../data/processed/test_preds.parquet", index=False)

In [3]:
import numpy as np

for thr in [-0.2, -0.1, 0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    p = np.where(test_df["vader_score"] >= thr, "positive", "negative")
    print(thr, round(accuracy_score(y_test, p), 4))

-0.2 0.6999
-0.1 0.71
0 0.7189
0.1 0.7298
0.2 0.7407
0.3 0.7512
0.4 0.7576
0.5 0.7635


In [4]:
import numpy as np

for thr in [0.5, 0.6, 0.7, 0.8, 0.9]:
    p = np.where(test_df["vader_score"] >= thr, "positive", "negative")
    print(thr, round(accuracy_score(y_test, p), 4))

0.5 0.7635
0.6 0.772
0.7 0.7707
0.8 0.762
0.9 0.7111


In [5]:
train_df = pd.read_parquet("../data/processed/train.parquet")
train_df["vader_score"] = train_df["text"].map(
    lambda t: analyzer.polarity_scores(t)["compound"]
)

best_thr, best_acc = None, 0
for thr in np.arange(-0.5, 0.91, 0.05):
    p = np.where(train_df["vader_score"] >= thr, "positive", "negative")
    acc = accuracy_score(train_df["sentiment"], p)
    if acc > best_acc:
        best_thr, best_acc = thr, acc

print("Best threshold on TRAIN:", round(best_thr, 2), "acc:", round(best_acc, 4))

p_test = np.where(test_df["vader_score"] >= best_thr, "positive", "negative")
print("Test accuracy with that threshold:", round(accuracy_score(y_test, p_test), 4))

Best threshold on TRAIN: 0.7 acc: 0.7642
Test accuracy with that threshold: 0.7707


In [6]:
test_df.to_parquet("../data/processed/test_preds.parquet", index=False)

In [7]:
test_df["pred_vader_tuned"] = np.where(test_df["vader_score"] >= 0.7, "positive", "negative")
test_df.to_parquet("../data/processed/test_preds.parquet", index=False)